### Raw Layer

The Raw layer holds the original source data in its unmodified form. Files are extracted from
the source archive and organized into a consistent directory structure, separated by entity
(customer, product, sales) and load type (historical, incremental). No transformation, cleaning,
or type casting is applied at this stage — Raw exists purely to establish a reliable, traceable
starting point for the pipeline, preserving exactly what was received from source.

### Landing Layer

The Landing layer converts Raw CSV files into Parquet format. Parquet is a columnar storage
format optimized for distributed processing, offering faster reads, better compression, and
schema awareness compared to plain CSV. This conversion improves performance for every
downstream layer without altering the underlying data. A row-count audit is performed after
conversion to confirm data integrity was preserved through the format change.

In [0]:
import zipfile

zip_path = "/Volumes/apex_retail/retail/source_files/Datasets-20260707T163414Z-3-001.zip"
extract_path = "/Volumes/apex_retail/retail/source_files/"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

In [0]:
import os
for root, dirs, files in os.walk(extract_path):
    print(root, files)

/Volumes/apex_retail/retail/source_files/ ['Datasets-20260707T163414Z-3-001.zip']
/Volumes/apex_retail/retail/source_files/Datasets []
/Volumes/apex_retail/retail/source_files/Datasets/audit_landing ['customer_incrementalaudit.csv', 'customer_historical_audit.csv', 'product_historical_audit.csv', 'sales_incrementalaudit.csv', 'sales_historical_audit.csv', 'product_incrementalaudit.csv']
/Volumes/apex_retail/retail/source_files/Datasets/incremental_data []
/Volumes/apex_retail/retail/source_files/Datasets/incremental_data/customer_incremental ['customer_incremental.csv']
/Volumes/apex_retail/retail/source_files/Datasets/incremental_data/sales_incremental ['sales_incremental.csv']
/Volumes/apex_retail/retail/source_files/Datasets/incremental_data/product_incremental ['product_incremental.csv']
/Volumes/apex_retail/retail/source_files/Datasets/audit_silver ['sales_incrementalaudit_silver.csv', 'product_silver_audit.csv', 'customer_incrementalaudit_silver.csv', 'customer_silver_audit.csv',

In [0]:
base = "/Volumes/apex_retail/retail/source_files/Datasets"
raw_base = "/Volumes/apex_retail/retail/source_files/raw"

# table_name -> (source_folder, filename)
sources = {
    ("customer", "historical"):   f"{base}/historical_data/customer/customer_historical.csv",
    ("product", "historical"):    f"{base}/historical_data/product/product_historical.csv",
    ("sales", "historical"):      f"{base}/historical_data/sales/sales_historical.csv",
    ("customer", "incremental"):  f"{base}/incremental_data/customer_incremental/customer_incremental.csv",
    ("product", "incremental"):   f"{base}/incremental_data/product_incremental/product_incremental.csv",
    ("sales", "incremental"):     f"{base}/incremental_data/sales_incremental/sales_incremental.csv",
}

raw_dfs = {}

for (table, load), path in sources.items():
    df = spark.read.option("header", True).csv(path)  # all columns land as String by default

    # the source customer_incremental file has a few unexpected extra columns
    # (surrogate_key, version, effective_start_date, effective_end_date, is_current)
    # that aren't part of the actual schema - dropping them here so raw stays clean
    if table == "customer" and load == "incremental":
        expected_customer_cols = [
            "customer_id", "age", "gender", "income_bracket", "loyalty_program",
            "membership_years", "churned", "marital_status", "number_of_children",
            "education_level", "occupation", "customer_zip_code", "customer_city", "customer_state"
        ]
        df = df.select(*expected_customer_cols)

    out_path = f"{raw_base}/{load}/{table}"
    df.write.mode("overwrite").csv(out_path, header=True)
    raw_dfs[(table, load)] = df
    print(f"{table} ({load}): {df.count()} rows -> {out_path}")

customer (historical): 1052 rows -> /Volumes/apex_retail/retail/source_files/raw/historical/customer
product (historical): 1043 rows -> /Volumes/apex_retail/retail/source_files/raw/historical/product
sales (historical): 1002 rows -> /Volumes/apex_retail/retail/source_files/raw/historical/sales
customer (incremental): 1053 rows -> /Volumes/apex_retail/retail/source_files/raw/incremental/customer
product (incremental): 1041 rows -> /Volumes/apex_retail/retail/source_files/raw/incremental/product
sales (incremental): 1000 rows -> /Volumes/apex_retail/retail/source_files/raw/incremental/sales


In [0]:
# ---- Phase 2: Landing ----

raw_base = "/Volumes/apex_retail/retail/source_files/raw"
landing_base = "/Volumes/apex_retail/retail/source_files/landing"
audit_landing_base = "/Volumes/apex_retail/retail/source_files/Datasets/audit_landing"

tables_loads = [
    ("customer", "historical"), ("customer", "incremental"),
    ("product", "historical"),  ("product", "incremental"),
    ("sales", "historical"),    ("sales", "incremental"),
]

audit_files = {
    ("customer", "historical"):  "customer_historical_audit.csv",
    ("customer", "incremental"): "customer_incrementalaudit.csv",
    ("product", "historical"):   "product_historical_audit.csv",
    ("product", "incremental"):  "product_incrementalaudit.csv",
    ("sales", "historical"):     "sales_historical_audit.csv",
    ("sales", "incremental"):    "sales_incrementalaudit.csv",
}

# Step 1:we check so we can confirm real column names before trusting the logic
spark.read.option("header", True).csv(f"{audit_landing_base}/customer_historical_audit.csv").show(truncate=False)

+-------------------+---------+
|table_name         |row_count|
+-------------------+---------+
|customer_historical|1052     |
+-------------------+---------+



In [0]:
# paths for phase 2
raw_base = "/Volumes/apex_retail/retail/source_files/raw"
landing_base = "/Volumes/apex_retail/retail/source_files/landing"
audit_path = "/Volumes/apex_retail/retail/source_files/Datasets/audit_landing"

# all 6 table/load combos i need to process
tables_loads = [
    ("customer", "historical"),
    ("customer", "incremental"),
    ("product", "historical"),
    ("product", "incremental"),
    ("sales", "historical"),
    ("sales", "incremental"),
]

# mapping to actual audit filenames
audit_files = {
    ("customer", "historical"):  "customer_historical_audit.csv",
    ("customer", "incremental"): "customer_incrementalaudit.csv",
    ("product", "historical"):   "product_historical_audit.csv",
    ("product", "incremental"):  "product_incrementalaudit.csv",
    ("sales", "historical"):     "sales_historical_audit.csv",
    ("sales", "incremental"):    "sales_incrementalaudit.csv",
}

In [0]:
# reading raw csvs and writing them out as parquet for landing
for table, load in tables_loads:
    raw_path = f"{raw_base}/{load}/{table}"
    df = spark.read.option("header", True).csv(raw_path)

    landing_out = f"{landing_base}/{load}/{table}"
    df.write.mode("overwrite").parquet(landing_out)

    print(f"{table} ({load}) -> parquet written, {df.count()} rows")

customer (historical) -> parquet written, 1052 rows
customer (incremental) -> parquet written, 1053 rows
product (historical) -> parquet written, 1043 rows
product (incremental) -> parquet written, 1041 rows
sales (historical) -> parquet written, 1002 rows
sales (incremental) -> parquet written, 1000 rows


In [0]:
# checks actual row count
def check_audit(actual_count, audit_file, table_key):
    audit_df = spark.read.option("header", True).csv(f"{audit_path}/{audit_file}")
    match = audit_df.filter(audit_df.table_name == table_key).collect()

    if len(match) == 0:
        raise Exception(f"couldn't find {table_key} in {audit_file}")

    expected = int(match[0]["row_count"])
    status = "PASS" if expected == actual_count else "FAIL"
    return expected, status

In [0]:
# now we actually run the validation for all 6 tables
audit_results = []

for table, load in tables_loads:
    landing_path = f"{landing_base}/{load}/{table}"
    df = spark.read.parquet(landing_path)
    actual = df.count()

    table_key = f"{table}_{load}"
    audit_file = audit_files[(table, load)]

    expected, status = check_audit(actual, audit_file, table_key)
    audit_results.append((table_key, expected, actual, status))

audit_report = spark.createDataFrame(audit_results, ["table_name", "expected_count", "actual_count", "status"])
audit_report.show(truncate=False)

# stop here if anything failed
failed = audit_report.filter(audit_report.status == "FAIL").count()
if failed > 0:
    raise Exception(f"{failed} table(s) failed audit validation, stopping pipeline")
else:
    print("all 6 tables passed landing audit")

+--------------------+--------------+------------+------+
|table_name          |expected_count|actual_count|status|
+--------------------+--------------+------------+------+
|customer_historical |1052          |1052        |PASS  |
|customer_incremental|1053          |1053        |PASS  |
|product_historical  |1043          |1043        |PASS  |
|product_incremental |1041          |1041        |PASS  |
|sales_historical    |1002          |1002        |PASS  |
|sales_incremental   |1000          |1000        |PASS  |
+--------------------+--------------+------------+------+

all 6 tables passed landing audit


In [0]:
raw_base = "/Volumes/apex_retail/retail/source_files/raw" 
landing_base = "/Volumes/apex_retail/retail/source_files/landing"

df = spark.read.option("header", True).csv(f"{raw_base}/incremental/customer")
print("Raw CSV columns:", df.columns)

df.write.mode("overwrite").parquet(f"{landing_base}/incremental/customer")
print("Landing customer incremental rebuilt, rows:", df.count())

Raw CSV columns: ['customer_id', 'age', 'gender', 'income_bracket', 'loyalty_program', 'membership_years', 'churned', 'marital_status', 'number_of_children', 'education_level', 'occupation', 'customer_zip_code', 'customer_city', 'customer_state']
Landing customer incremental rebuilt, rows: 1053


In [0]:
display(dbutils.fs.ls("/Volumes/apex_retail/retail/source_files/raw/incremental/customer/"))

path,name,size,modificationTime
dbfs:/Volumes/apex_retail/retail/source_files/raw/incremental/customer/_committed_1069786470978623253,_committed_1069786470978623253,201,1783799472000
dbfs:/Volumes/apex_retail/retail/source_files/raw/incremental/customer/_committed_3529211210663106040,_committed_3529211210663106040,212,1783774817000
dbfs:/Volumes/apex_retail/retail/source_files/raw/incremental/customer/_committed_8345459484398379841,_committed_8345459484398379841,201,1783799004000
dbfs:/Volumes/apex_retail/retail/source_files/raw/incremental/customer/_started_1069786470978623253,_started_1069786470978623253,0,1783799472000
dbfs:/Volumes/apex_retail/retail/source_files/raw/incremental/customer/_started_8345459484398379841,_started_8345459484398379841,0,1783799004000
dbfs:/Volumes/apex_retail/retail/source_files/raw/incremental/customer/part-00000-tid-1069786470978623253-a6d98002-bd62-46fe-8eb1-ecbd536503fa-445-1-c000.csv,part-00000-tid-1069786470978623253-a6d98002-bd62-46fe-8eb1-ecbd536503fa-445-1-c000.csv,82160,1783799472000


### Summary

This notebook covers the Raw and Landing phases of the pipeline - the first two steps in
getting source data ready for Delta Lake processing.

**Raw:** Source CSV files were read from the provided dataset and organized into a raw/
directory, split by load type (historical, incremental) and entity (customer, product, sales)
- 6 files total. All columns were kept in string format, with no transformation applied.

During development, one source file (customer_incremental.csv) was found to contain a few
unexpected extra columns not part of the documented schema. An explicit column selection was
added during Raw ingestion to filter these out, ensuring only the genuine 14 source fields are
carried into the rest of the pipeline.

**Landing:** Raw CSVs were converted into Parquet format, a columnar storage format that
speeds up reads for every downstream layer. All 6 files were converted with string formatting
preserved.

Row counts were validated against the audit_landing files provided with the source data - each
file's declared expected count was read dynamically and compared against the actual converted
row count. All 6 tables passed validation with exact matches; the pipeline halts and raises an
exception if any table fails this check.

**Output:** 6 Parquet datasets under `/Volumes/apex_retail/retail/source_files/landing/`,
ready for Delta Lake conversion in the Bronze phase.